In [7]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict,Annotated
import os
import operator
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import Literal
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

True

In [8]:
endpoint=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    task="text-generation"
    )
model=ChatHuggingFace(llm=endpoint)

In [9]:
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]


In [10]:
def chat(state:ChatState):
    # take user query from state
    messages=state['messages']
    # send to llm
    response=model.invoke(messages)
    # response store state
    return {'messages':[response]}

In [11]:
checkpoint=MemorySaver()
graph=StateGraph(ChatState)

graph.add_node("chat",chat)

graph.add_edge(START,"chat")
graph.add_edge("chat",END)

chatbot=graph.compile(checkpointer=checkpoint)

In [ ]:
thread_id='1'
while True:
    user_input=input("You: ")
    if user_input.strip().lower() in ['exit','quit','bye']:
        break

    config={'configurable':{'thread_id':thread_id}}
    state=chatbot.invoke({'messages':[HumanMessage(content=user_input)]},config=config)

    print('AI:',state['messages'][-1].content)

In [ ]:
initial_state={
    'messages':[HumanMessage(content="Hello, how are you?")]
}
chatbot.invoke(initial_state)